# Fase 1 — Reproducir el repo oficial sobre DIVA-HisDB

Este notebook **no contiene logica**: solo orquesta el paquete `backend/paper_replica/`,
que esta cubierto por tests. Si algo falla aca, se arregla en el paquete y se agrega
el test correspondiente — no se parchea el notebook.

- Diseno: `docs/replica-denardin/2026-08-10-diseno.md`
- Plan: `docs/replica-denardin/2026-08-10-plan-fase-1.md`

**Criterio de aceptacion:** F1 dentro de ±0.02 de CB55 0.991 / CS18 0.984 / CS863 0.975.
Si no se cumple, no se avanza a la Fase 2.

**Runtime:** Entorno de ejecucion → Cambiar tipo de entorno → **T4 GPU**.

## 1. Entorno y datos

In [ ]:
!pip -q install segmentation-models-pytorch==0.3.4 einops==0.8.0 scikit-image==0.24.0

!git clone --depth 1 -b develop https://github.com/gabrielaz/Etiquetado.git /content/repo
!git clone --depth 1 https://github.com/axelden/Few-shot-DIA-WACV2023.git /content/datos-diva

import sys
sys.path.insert(0, '/content/repo/backend')

El paquete busca los datos en `<repo>/../datos-diva`. En Colab el repo esta en
`/content/repo`, asi que `DIVA_ROOT` apunta a `/content/datos-diva/fewshot_data`.

In [ ]:
from paper_replica.config import DIVA_ROOT, MANUSCRITOS, Config

print('DIVA_ROOT:', DIVA_ROOT)
assert DIVA_ROOT.exists(), f'no encuentro los datos en {DIVA_ROOT}'

for m in MANUSCRITOS:
    for split in ('training', 'validation', 'public-test'):
        n = len(list((DIVA_ROOT / m / f'img-{m}' / split).glob('*')))
        print(f'  {m:6} {split:12} {n:2d} paginas')

## 2. Los tests corren primero

Antes de gastar GPU. Si algo esta roto, se sabe aca y no despues de 200 epocas.

In [ ]:
!cd /content/repo/backend && python -m pytest paper_replica/tests/ -q -k "not end_to_end"

## 3. GPU y configuracion

In [ ]:
import torch

assert torch.cuda.is_available(), 'activa la GPU: Entorno de ejecucion -> Cambiar tipo de entorno -> T4'
props = torch.cuda.get_device_properties(0)
print(f'{props.name}  {props.total_memory / 1e9:.1f} GB')

cfg = Config()
print(f'parche {cfg.patch_size}  grilla {cfg.grid_cols}x{cfg.grid_rows}='
      f'{cfg.patches_per_page}  +{cfg.n_crops} crops  micro_batch {cfg.micro_batch}')
print(f'backbone {cfg.backbone}  pesos {cfg.encoder_weights}  AMP {cfg.use_amp}')

## 4. Entrenamiento

Tres manuscritos, hasta 200 epocas cada uno con early stopping (paciencia 20,
minimo 50 epocas). Los checkpoints se guardan en `/content/`.

Si aparece OOM: bajar `cfg = Config(micro_batch=4)`. **Nunca** por debajo de 4, y
nunca compensar congelando BatchNorm ni activando AMP — ese fue el error del
intento anterior.

In [ ]:
import json, time
from paper_replica.train import entrenar

resultados = {}
for m in MANUSCRITOS:
    t0 = time.time()
    resultados[m] = entrenar(m, cfg, device='cuda', log_cada=5)
    torch.save(resultados[m]['state_dict'], f'/content/{m}.pt')
    with open(f'/content/{m}_historial.json', 'w') as f:
        json.dump(resultados[m]['historial'], f, indent=2)
    print(f'{m}: mejor epoca {resultados[m]["mejor_epoca"]}  '
          f'val {resultados[m]["mejor_val_loss"]:.4f}  '
          f'{(time.time() - t0) / 60:.1f} min\n')
    torch.cuda.empty_cache()

## 5. Evaluacion y veredicto

Se reportan las dos lecturas del F1 del paper (ver diseno §8). Alcanza con que
**una** reproduzca los numeros publicados: eso mismo responde cual usaron los autores.

In [ ]:
from paper_replica.report import evaluar_manuscrito, imprimir_veredicto

evaluaciones = [
    evaluar_manuscrito(m, resultados[m]['state_dict'], cfg, device='cuda')
    for m in MANUSCRITOS
]
imprimir_veredicto(evaluaciones)

In [ ]:
for e in evaluaciones:
    print(f"=== {e['manuscrito']} ===")
    print('  variante A:', {k: round(v, 4) for k, v in e['f1_variante_a'].items()})
    print('  variante B:', {k: round(v, 4) for k, v in e['f1_variante_b'].items()})
    print('  max logit :', f"{e['max_logit']:.1f}")
    print('  confusion (filas=gt, columnas=pred):')
    for fila in e['confusion']:
        print('   ', '  '.join(f'{v:>10d}' for v in fila))
    print()

## 6. Ablacion del refinamiento

Responde dos preguntas abiertas del diseno: cuanto aporta Sauvola, y si el `k`
correcto es el del codigo oficial (0.01) o el del paper 2024 (0.2).

In [ ]:
from dataclasses import replace

print(f"{'manuscrito':<10} {'refinamiento':<16} {'media A':>9} {'media B':>9}")
print('-' * 48)
for m in MANUSCRITOS:
    sd = resultados[m]['state_dict']
    variantes = [
        ('sin refinar', replace(cfg), False),
        ('sauvola k=0.01', replace(cfg, sauvola_k=0.01), True),
        ('sauvola k=0.2', replace(cfg, sauvola_k=0.2), True),
    ]
    for nombre, c, refinar_flag in variantes:
        r = evaluar_manuscrito(m, sd, c, device='cuda', refinamiento=refinar_flag)
        print(f"{m:<10} {nombre:<16} {r['media_a']:>9.4f} {r['media_b']:>9.4f}")

## 7. Que hacer con el resultado

**Si pasa:** anotar los numeros en `docs/replica-denardin/resultados-fase-1.md`
y avanzar a la Fase 2 (configuracion del paper 2024 sobre los mismos datos).

**Si no pasa:** no avanzar. Sospechosos en orden, todos localizables porque los
datos y los hiperparametros son los oficiales:

1. El `softmax` espurio del codigo oficial antes de la CE — probar replicando el bug
2. `k` de Sauvola y el refinamiento (celda 6)
3. `min_size` de `removeSmallCC`, fijado en 64 sin referencia del oficial
4. La interpolacion del resize (bilineal imagen / vecino mas cercano GT)
5. `micro_batch` afectando a BatchNorm — comparar 8 contra 4